In [1]:
# %% [markdown]
# # Fast website category classifier from raw HTML
#
# Fastest production-oriented approach:
# - parse raw HTML with `selectolax`
# - extract useful text fields
# - train a `fastText` multiclass classifier
#
# Categories:
# - Adult
# - Finance
# - Computers
# - Health
# - Entertainment
# - News
# - Shopping
# - Sports
# - Food
# - Travel
# - Automotive
#
# Expected training CSV columns:
# - `html`
# - `label`

# %%
%pip install -q pandas scikit-learn selectolax fasttext-wheel joblib tqdm

# %%
from __future__ import annotations

import os
import re
import json
import gzip
import base64
import html as ihtml
import importlib
from pathlib import Path

import fasttext
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from selectolax.parser import HTMLParser
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

tqdm.pandas()

# fasttext 0.9.x calls np.array(..., copy=False), which breaks on NumPy 2.x.
# Keep the workaround local to fasttext instead of changing NumPy globally.
try:
    _fasttext_impl = importlib.import_module("fasttext.FastText")
    _fasttext_np_array = _fasttext_impl.np.array

    def _fasttext_np_array_compat(*args, **kwargs):
        kwargs.pop("copy", None)
        return _fasttext_np_array(*args, **kwargs)

    _fasttext_impl.np.array = _fasttext_np_array_compat
except Exception:
    pass

# %%
CATEGORIES = [
    "Adult",
    "Finance",
    "Computers",
    "Health",
    "Entertainment",
    "News",
    "Shopping",
    "Sports",
    "Food",
    "Travel",
    "Automotive",
]

CATEGORY_SET = set(CATEGORIES)
CATEGORY_LOOKUP = {category.lower(): category for category in CATEGORIES}

DATA_PATH = Path("D:\\thor_dataset\datasets\\15_07_1.jsonl")  # supports CSV(html,label) and JSONL(compressed_html,categories)
MODEL_PATH = Path("website_classifier.bin")
QUANT_MODEL_PATH = Path("website_classifier.ftz")

# %%
# Fast regexes compiled once
RE_WS = re.compile(r"\s+")
RE_BAD = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")
RE_LABEL_SAFE = re.compile(r"[^A-Za-z0-9]+")

# %%
def normalize_text(text: str) -> str:
    text = ihtml.unescape(text or "")
    text = RE_BAD.sub(" ", text)
    text = RE_WS.sub(" ", text).strip()
    return text


def safe_label(label: str) -> str:
    label = label.strip()
    if label not in CATEGORY_SET:
        raise ValueError(f"Unknown label: {label}")
    return f"__label__{RE_LABEL_SAFE.sub('_', label)}"


def unsafify_label(ft_label: str) -> str:
    return ft_label.replace("__label__", "").replace("_", " ")


def decompress_html(value: str) -> str:
    if value is None:
        return ""
    if not isinstance(value, str):
        value = str(value)
    value = value.strip()
    if not value:
        return ""
    if "<html" in value.lower() or "<body" in value.lower() or "<div" in value.lower():
        return value

    try:
        decoded = base64.b64decode(value)
    except Exception:
        return value

    try:
        return gzip.decompress(decoded).decode("utf-8", errors="ignore")
    except Exception:
        return decoded.decode("utf-8", errors="ignore")


def parse_label(value: object) -> str | None:
    if value is None:
        return None

    if isinstance(value, list):
        candidates = value
    elif isinstance(value, str):
        raw = value.strip()
        if not raw:
            return None
        if raw.startswith("["):
            try:
                parsed = json.loads(raw)
                candidates = parsed if isinstance(parsed, list) else [raw]
            except json.JSONDecodeError:
                candidates = re.split(r"[,;|]", raw)
        else:
            candidates = re.split(r"[,;|]", raw)
    else:
        candidates = [str(value)]

    for candidate in candidates:
        key = str(candidate).strip().lower()
        if not key:
            continue
        mapped = CATEGORY_LOOKUP.get(key)
        if mapped:
            return mapped
    return None


def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f"Skipping invalid JSON on line {line_no}.")
        df = pd.DataFrame(records)
    else:
        df = pd.read_csv(path)

    if {"html", "label"}.issubset(df.columns):
        pass
    elif {"compressed_html", "categories"}.issubset(df.columns):
        df = df.copy()
        df["html"] = df["compressed_html"].apply(decompress_html)
        df["label"] = df["categories"].apply(parse_label)
    else:
        raise ValueError(
            "Input must contain either (html,label) or (compressed_html,categories)."
        )

    df = df.dropna(subset=["html", "label"]).copy()
    df["html"] = df["html"].astype(str)
    df["label"] = df["label"].apply(parse_label)
    df = df.dropna(subset=["label"]).copy()
    return df


# %%
def extract_text_from_html(
    raw_html: str,
    max_body_chars: int = 4000,
    ) -> str:
    """
    Fast HTML -> text extraction.

    Strategy:
    - remove noisy tags
    - pull title/meta/og/headings
    - pull main visible text
    - weight title/headings slightly by repetition
    """
    if not raw_html or not isinstance(raw_html, str):
        return ""

    tree = HTMLParser(raw_html)
    if tree is None:
        return ""

    for tag in [
        "script",
        "style",
        "noscript",
        "svg",
        "canvas",
        "iframe",
        "footer",
        "nav",
        "form",
        "aside",
    ]:
        for node in tree.css(tag):
            node.decompose()

    parts = []

    title = tree.css_first("title")
    if title:
        t = normalize_text(title.text())
        if t:
            parts.extend([t, t, t])

    meta_desc = tree.css_first('meta[name="description"]')
    if meta_desc:
        v = normalize_text(meta_desc.attributes.get("content", ""))
        if v:
            parts.extend([v, v])

    og_title = tree.css_first('meta[property="og:title"]')
    if og_title:
        v = normalize_text(og_title.attributes.get("content", ""))
        if v:
            parts.extend([v, v])

    og_desc = tree.css_first('meta[property="og:description"]')
    if og_desc:
        v = normalize_text(og_desc.attributes.get("content", ""))
        if v:
            parts.append(v)

    for sel in ["h1", "h2", "h3"]:
        texts = []
        for node in tree.css(sel):
            txt = normalize_text(node.text())
            if txt:
                texts.append(txt)
        if texts:
            joined = " ".join(texts[:20])
            parts.extend([joined, joined])

    body = tree.body
    if body:
        body_text = normalize_text(body.text(separator=" "))
        if body_text:
            parts.append(body_text[:max_body_chars])

    text = normalize_text(" ".join(parts))
    return text


# %%
def build_ft_line(html_text: str, label: str) -> str:
    text = extract_text_from_html(html_text)
    text = normalize_text(text.lower())
    return f"{safe_label(label)} {text}"


def build_ft_line_unlabeled(html_text: str) -> str:
    text = extract_text_from_html(html_text)
    return normalize_text(text.lower())


# %% [markdown]
# ## Load data
#
# Supported input formats:
# - CSV with columns: `html`, `label`
# - JSONL with columns: `compressed_html`, `categories`

# %%
df = load_dataset(DATA_PATH)

bad_labels = sorted(set(df["label"]) - CATEGORY_SET)
if bad_labels:
    raise ValueError(f"Unexpected labels in input: {bad_labels}")

print(df.shape)
df.head()

# %% [markdown]
# ## Optional: inspect label distribution

# %%
df["label"].value_counts()

# %% [markdown]
# ## Train/validation split

# %%
train_df, valid_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"],
)

print("train:", train_df.shape)
print("valid:", valid_df.shape)

# %% [markdown]
# ## Convert HTML into fastText training format
#
# We parallelize extraction because raw HTML parsing is the slowest step.

# %%
def parallel_build_lines(frame: pd.DataFrame, n_jobs: int = -1) -> list[str]:
    rows = frame[["html", "label"]].itertuples(index=False, name=None)
    return Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(build_ft_line)(html, label) for html, label in rows
    )


train_lines = parallel_build_lines(train_df)
valid_lines = parallel_build_lines(valid_df)

# %%
train_txt = Path("train.ft.txt")
valid_txt = Path("valid.ft.txt")

train_txt.write_text("\n".join(train_lines), encoding="utf-8")
valid_txt.write_text("\n".join(valid_lines), encoding="utf-8")

print(train_txt, train_txt.stat().st_size)
print(valid_txt, valid_txt.stat().st_size)

# %% [markdown]
# ## Train fastText
#
# Notes:
# - `wordNgrams=2` usually helps website/category classification
# - `minn/maxn` adds character n-grams, useful for noisy web text
# - `dim=64` is a good speed/quality compromise
# - `epoch` can be increased if you have more data and want more accuracy

# %%
model = fasttext.train_supervised(
    input=str(train_txt),
    lr=0.5,
    epoch=20,
    wordNgrams=2,
    dim=64,
    minn=2,
    maxn=5,
    bucket=2_000_000,
    loss="softmax",
    thread=os.cpu_count() or 4,
 )

model.save_model(str(MODEL_PATH))
print(f"Saved: {MODEL_PATH}")

# %% [markdown]
# ## Validate

# %%
result = model.test(str(valid_txt))
print("N examples:", result[0])
print("Precision@1:", result[1])
print("Recall@1:", result[2])

# %%
def predict_label_and_scores(
    raw_html: str,
    model: fasttext.FastText._FastText,
    k: int = 3,
 ):
    text = build_ft_line_unlabeled(raw_html)
    labels, probs = model.predict(text, k=k)
    return [
        {
            "label": unsafify_label(label),
            "score": float(prob),
        }
        for label, prob in zip(labels, probs)
    ]


# %%
y_true = valid_df["label"].tolist()
y_pred = []

for html in valid_df["html"].tolist():
    pred = predict_label_and_scores(html, model, k=1)[0]["label"]
    y_pred.append(pred)

print(classification_report(y_true, y_pred, digits=4))

# %%
cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=CATEGORIES),
    index=CATEGORIES,
    columns=CATEGORIES,
)
cm

# %% [markdown]
# ## Quantize model for faster/lighter deployment
#
# This usually makes the model much smaller and can improve real-world latency.

# %%
model.quantize(
    input=str(train_txt),
    retrain=True,
    cutoff=50_000,
    thread=os.cpu_count() or 4,
 )
model.save_model(str(QUANT_MODEL_PATH))
print(f"Saved quantized model: {QUANT_MODEL_PATH}")

# %%
quant_model = fasttext.load_model(str(QUANT_MODEL_PATH))

# %% [markdown]
# ## Single prediction API

# %%
def classify_html(
    raw_html: str,
    model: fasttext.FastText._FastText = quant_model,
    k: int = 3,
 ):
    preds = predict_label_and_scores(raw_html, model=model, k=k)
    top1 = preds[0]
    return {
        "label": top1["label"],
        "confidence": top1["score"],
        "top_k": preds,
    }


# %%
sample_html = """
<html>
  <head>
    <title>Latest Premier League Results and Match Analysis</title>
    <meta name="description" content="Scores, standings, fixtures, and football news." />
  </head>
  <body>
    <h1>Premier League roundup</h1>
    <p>Match highlights, player ratings, and standings update.</p>
  </body>
</html>
"""

classify_html(sample_html)

# %% [markdown]
# ## Batch inference from raw HTML
#
# Input: iterable of HTML strings
#
# Output: DataFrame with label + confidence

# %%
def classify_many(
    html_list: list[str],
    model: fasttext.FastText._FastText = quant_model,
    k: int = 3,
 ) -> pd.DataFrame:
    texts = [build_ft_line_unlabeled(x) for x in html_list]
    labels, probs = model.predict(texts, k=k)

    rows = []
    for i in range(len(html_list)):
        top_k = [
            {
                "label": unsafify_label(lbl),
                "score": float(scr),
            }
            for lbl, scr in zip(labels[i], probs[i])
        ]
        rows.append(
            {
                "label": top_k[0]["label"],
                "confidence": top_k[0]["score"],
                "top_k": top_k,
            }
        )

    return pd.DataFrame(rows)


# %%
sample_batch = [
    """
    <html><head><title>Best laptop CPUs in 2026</title></head>
    <body><h1>Desktop vs laptop processors</h1><p>Benchmarks and PC builds</p></body></html>
    """
    """
    <html><head><title>Cheap flights to Rome</title></head>
    <body><h1>Travel deals</h1><p>Hotels, itineraries, and city guides</p></body></html>
    """
    """
    <html><head><title>Car insurance comparison</title></head>
    <body><h1>Auto coverage plans</h1><p>Compare quotes and premiums</p></body></html>
    """
]

classify_many(sample_batch)

# %% [markdown]
# ## Deployment helper
#
# Load the quantized model once per process and reuse it.

# %%
DEPLOY_MODEL = fasttext.load_model(str(QUANT_MODEL_PATH))


def predict_category(raw_html: str) -> tuple[str, float]:
    text = build_ft_line_unlabeled(raw_html)
    labels, probs = DEPLOY_MODEL.predict(text, k=1)
    return unsafify_label(labels[0]), float(probs[0])


# %%
label, score = predict_category(sample_html)
print(label, score)

Note: you may need to restart the kernel to use updated packages.
(6511, 5)
train: (5534, 5)
valid: (977, 5)
train.ft.txt 28909933
valid.ft.txt 5099736
Saved: website_classifier.bin
N examples: 977
Precision@1: 0.691914022517912
Recall@1: 0.691914022517912
               precision    recall  f1-score   support

        Adult     0.8333    0.6250    0.7143        16
   Automotive     1.0000    0.0357    0.0690        28
    Computers     0.7512    0.8556    0.8000       180
Entertainment     0.6387    0.7308    0.6816       104
      Finance     0.6337    0.6275    0.6305       102
         Food     0.6875    0.2750    0.3929        40
       Health     0.5484    0.4722    0.5075        72
         News     0.6667    0.7717    0.7153       127
     Shopping     0.7162    0.8200    0.7646       200
       Sports     0.8636    0.4524    0.5938        42
       Travel     0.7143    0.6818    0.6977        66

     accuracy                         0.6919       977
    macro avg     0.7321  